### Résultat éval
microsoft/deberta-v3-large, matous-volf/political-leaning-deberta-large : 0.725925925925926, 0.5934065934065934 (English) ; 0.625, 0.559 (Both) ; 0.599, 0.552 (Français)



sentence bert + XGB = 0.5653452685421995 de précision macro, f1_macro = 0.5660843021367798 ; Confusion matrix = [[61 27], [24 19]]

In [3]:
import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

# Matrice de confusion
cm = np.array([[70, 19],
       [18, 28]])

# Reconstruction de y_test et y_pred à partir de la matrice
TN, FP, FN, TP = cm.ravel()

y_test = (
    [0] * TN + [0] * FP +
    [1] * FN + [1] * TP
)

y_pred = (
    [0] * TN + [1] * FP +
    [0] * FN + [1] * TP
)

# Métriques par classe
precision_per_class = precision_score(y_test, y_pred, average=None)
recall_per_class = recall_score(y_test, y_pred, average=None)
f1_per_class = f1_score(y_test, y_pred, average=None)

# Support par classe
support_class_0 = TN + FP
support_class_1 = FN + TP

# Moyennes macro
precision_macro = precision_score(y_test, y_pred, average='macro')
recall_macro = recall_score(y_test, y_pred, average='macro')
f1_macro = f1_score(y_test, y_pred, average='macro')

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

# Affichage
for i in range(2):
    support = support_class_0 if i == 0 else support_class_1

    print(f"Classe {i}")
    print(f"Precision : {precision_per_class[i]:.4f}")
    print(f"Recall    : {recall_per_class[i]:.4f}")
    print(f"F1-score  : {f1_per_class[i]:.4f}")
    print(f"Support   : {support}")
    print()

print("=== Moyennes macro ===")
print(f"Precision macro : {precision_macro:.4f}")
print(f"Recall macro    : {recall_macro:.4f}")
print(f"F1 macro        : {f1_macro:.4f}")

print("\n=== Accuracy ===")
print(f"Accuracy : {accuracy:.4f}")

Classe 0
Precision : 0.7955
Recall    : 0.7865
F1-score  : 0.7910
Support   : 89

Classe 1
Precision : 0.5957
Recall    : 0.6087
F1-score  : 0.6022
Support   : 46

=== Moyennes macro ===
Precision macro : 0.6956
Recall macro    : 0.6976
F1 macro        : 0.6966

=== Accuracy ===
Accuracy : 0.7259


In [ ]:
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import torch

from sentence_transformers import SentenceTransformer

from xgboost import XGBClassifier

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
MODEL_NAME = (
    "sentence-transformers/"
    "paraphrase-multilingual-mpnet-base-v2"
)

sbert = SentenceTransformer(MODEL_NAME,
    device=device)

In [ ]:
df = pd.read_csv('sample_annotated.csv')
df = df[df['annotation']!='Unclassifiable']
len(df)

In [ ]:
import re
import unicodedata
import html

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    # 1. HTML entities (&gt etc.)
    text = html.unescape(text)

    # 2. Unicode normalization
    text = unicodedata.normalize("NFKC", text)

    # 3. Fix escaped apostrophes (IMPORTANT)
    text = text.replace("\\'", "'")

    # 4. Remove leftover backslashes
    text = text.replace("\\", "")

    # 5. Fix non-breaking spaces
    text = text.replace("\xa0", " ")

    # 6. Collapse spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [ ]:
texts = df['text'].astype(str).apply(clean_text).tolist()

X = sbert.encode(
    texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype(np.float32)


print("Embeddings shape:", X.shape)

In [ ]:
df['num_annotation'] = [0 if l=='Left' else 1 for l in df['annotation']]
y = df['num_annotation'].values.astype(np.int32)

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

def objective(trial):

    n_neg = np.sum(y == 0)
    n_pos = np.sum(y == 1)

    imbalance_ratio = n_neg / n_pos

    params = {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            800
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.5,
            log=True
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0,
            5
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            0,
            5
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            0.1,
            10
        ),

        "scale_pos_weight": trial.suggest_float(
            "scale_pos_weight",
            max(0.5, imbalance_ratio * 0.5),
            imbalance_ratio * 2
        ),

        "objective": "binary:logistic",

        "eval_metric": "logloss",

        "tree_method": "hist",

        "random_state": 42
    }

    threshold = trial.suggest_float(
        "threshold",
        0.1,
        0.9
    )

    fold_scores = []

    for train_idx, valid_idx in cv.split(X, y):

        X_train, X_valid = X[train_idx], X[valid_idx]

        y_train, y_valid = y[train_idx], y[valid_idx]

        model = XGBClassifier(**params)

        model.fit(X_train, y_train)

        y_prob = model.predict_proba(X_valid)[:, 1]

        y_pred = (
            y_prob >= threshold
        ).astype(int)
        
        precision = f1_score(
            y_valid,
            y_pred,
            average="macro",
            #zero_division=0
            )

        fold_scores.append(precision)

    return np.mean(fold_scores)

In [ ]:
optuna.logging.set_verbosity(
    optuna.logging.WARNING
)

study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=2000,
    show_progress_bar=True
)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
#best_params = {'n_estimators': 621, 'max_depth': 4, 'learning_rate': 0.02797921228582913, 'subsample': 0.6794447472202899, 'colsample_bytree': 0.6398935232339933, 'min_child_weight': 9, 'gamma': 0.6722581737436195, 'reg_alpha': 0.32830995249988104, 'reg_lambda': 9.172072886573819, 'threshold': 0.7368423591956468}
best_params = {'n_estimators': 579, 'max_depth': 10, 'learning_rate': 0.18767857646049085, 'subsample': 0.651887963753939, 'colsample_bytree': 0.6208042297903424, 'min_child_weight': 9, 'gamma': 1.0490747434777725, 'reg_alpha': 2.8491942460956086, 'reg_lambda': 8.068984282856123, 'scale_pos_weight': 2.9663750683596453, 'threshold': 0.5289310689146067}
best_treshold = best_params.pop('threshold')
final_model = XGBClassifier(
    **best_params,

    objective="binary:logistic",

    eval_metric="logloss",

    tree_method="hist",

    random_state=42
)

final_model.fit(X_train, y_train)

def predict_with_threshold(model, X, threshold=0.5):

    proba = model.predict_proba(X)[:, 1]

    return (proba >= threshold).astype(int)

y_pred = predict_with_threshold(final_model, X_test, best_treshold)

In [ ]:
precision_score(y_test, y_pred, average='macro')

In [ ]:
cm = confusion_matrix(
    y_test,
    y_pred
)
print(cm)